# Train a small Driver policy with TRL on the Global Logistics Resolver

This Colab fine-tunes a small instruction model (default: **Qwen2.5-0.5B-Instruct**, fits on a T4)
to behave as a per-truck `Driver` for the Global Logistics Resolver OpenEnv environment.

**Pipeline overview (Run All runs end-to-end and emits every artifact below):**

1. Install `trl`, `transformers`, `peft`, `accelerate`, `requests`, `matplotlib`.
2. Generate **trajectories** from the live env (HF Space) using a teacher policy.
3. **Score histogram** of every trajectory per task → `trajectory_score_hist.png` + `trajectory_score_summary.json`. The `keep threshold = 0.7` line shows exactly what survives the filter.
4. Filter trajectories by terminal `/grader` score (`>= 0.7`) and build the SFT chat dataset.
5. **SFT** a small student `Driver` on the filtered (state → action) JSON pairs (LoRA on a T4) → `sft_loss_curve.png`.
6. **Baseline vs trained** bar chart: mean terminal `/grader` score per task for the student **before vs after** LoRA → `baseline_vs_trained_grader.png` + `grader_eval_summary.json`.
7. **GRPO** loop with `GET /grader` as the reward function — the deck-recommended "training loop talks to env, not a static dataset" pattern. Default `GRPO_DRYRUN=0` runs the real ~20-step loop → `grpo_reward_curve.png` + `grpo_reward_summary.json` (set `GRPO_DRYRUN=1` for a wiring-only smoke test).
8. **3-seed scripted vs single-agent vs multi-agent comparison** against the same live env → `baseline_vs_multi.png` + `rollout_summary.json`. This is the README headline chart showing what hierarchy + inter-agent messaging buy at inference time.
9. One-click **download helper** pulls every artifact to your laptop, ready to copy into `docs/plots/`.

Every plotting cell both **saves to disk and renders inline**, so the notebook doubles as a reproducible run book for judges.

In [16]:
!pip -q install "trl>=0.11.0" "transformers>=4.44" "peft>=0.12" "accelerate>=0.33" "datasets>=2.20" requests matplotlib

In [ ]:
import json
import os
import random
import time
from pathlib import Path

import requests
from huggingface_hub import login

OPENENV_BASE_URL = os.environ.get("OPENENV_BASE_URL", "https://nikitha04-openenv-logistics.hf.space")
TEACHER_MODEL = os.environ.get("TEACHER_MODEL", "gpt-4o-mini")
STUDENT_MODEL = os.environ.get("STUDENT_MODEL", "Qwen/Qwen2.5-0.5B-Instruct")
NUM_ROLLOUTS = int(os.environ.get("NUM_ROLLOUTS", "6"))
SCORE_KEEP_THRESHOLD = float(os.environ.get("SCORE_KEEP", "0.7"))
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR", "/content/openenv-driver-sft"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)

HEADERS = {}
if HF_TOKEN:
    # Required if the target Space is private.
    HEADERS["Authorization"] = f"Bearer {HF_TOKEN}"


def checked_json(resp, name: str):
    ct = resp.headers.get("content-type", "")
    body_preview = (resp.text or "")[:300]
    if resp.status_code != 200:
        raise RuntimeError(f"[{name}] HTTP {resp.status_code} | ct={ct} | body={body_preview}")
    if "application/json" not in ct.lower():
        raise RuntimeError(f"[{name}] Non-JSON response | ct={ct} | body={body_preview}")
    try:
        return resp.json()
    except Exception as exc:
        raise RuntimeError(f"[{name}] JSON parse failed | body={body_preview}") from exc


health_resp = requests.get(f"{OPENENV_BASE_URL}/health", headers=HEADERS, timeout=30)
print("health:", health_resp.status_code, health_resp.text[:200])
assert health_resp.status_code == 200, (
    f"Env server is unreachable at {OPENENV_BASE_URL}/health (status={health_resp.status_code})."
)
print("env ok:", OPENENV_BASE_URL)

health: 200 {"status":"healthy","incident_id":"INC-2026-EAS-01"}
env ok: https://nikitha04-openenv-logistics.hf.space


## 1. Generate trajectories

We collect (system prompt, telemetry, action) tuples from the env using a teacher policy.
For the stub we use a deterministic scripted teacher; in the real run replace `teacher_action(...)`
with `agents.driver.Driver` (which calls your LLM via LiteLLM).

In [11]:
SCRIPTED = {
    "easy": [
        {"action_type": "check_network"},
        {"action_type": "load_truck", "truck_id": "T101", "warehouse": "North", "amount": 20},
        {"action_type": "route_truck", "truck_id": "T101", "route_id": "North_to_South"},
        {"action_type": "wait", "hours": 5},
        {"action_type": "load_truck", "truck_id": "T101", "warehouse": "South", "amount": 20},
        {"action_type": "route_truck", "truck_id": "T101", "route_id": "South_to_East"},
        {"action_type": "wait", "hours": 8},
    ],
    "medium": [
        {"action_type": "check_network"},
        {"action_type": "load_truck", "truck_id": "T101", "warehouse": "North", "amount": 50},
        {"action_type": "route_truck", "truck_id": "T101", "route_id": "North_to_East"},
        {"action_type": "wait", "hours": 10},
    ],
    "hard": [
        {"action_type": "check_network"},
        {"action_type": "load_truck", "truck_id": "T101", "warehouse": "North", "amount": 40},
        {"action_type": "route_truck", "truck_id": "T101", "route_id": "North_to_East"},
        {"action_type": "route_truck", "truck_id": "T102", "route_id": "South_to_North"},
        {"action_type": "wait", "hours": 5},
        {"action_type": "load_truck", "truck_id": "T102", "warehouse": "North", "amount": 10},
        {"action_type": "route_truck", "truck_id": "T102", "route_id": "North_to_South"},
        {"action_type": "wait", "hours": 10},
    ],
}


def teacher_action(task: str, step_idx: int):
    plan = SCRIPTED.get(task, [])
    return plan[step_idx] if step_idx < len(plan) else {"action_type": "wait", "hours": 1}


def collect_one_trajectory(task: str):
    reset_resp = checked_json(
        requests.post(
            f"{OPENENV_BASE_URL}/reset",
            json={"task_level": task},
            headers=HEADERS,
            timeout=30,
        ),
        "reset",
    )
    state0 = reset_resp.get("state", {})
    brief = state0.get("mission_brief", "")
    constraints = state0.get("instruction_constraints", [])
    samples = []
    step_idx = 0
    while True:
        public = checked_json(
            requests.get(f"{OPENENV_BASE_URL}/state", headers=HEADERS, timeout=30),
            "state",
        ).get("state", {})
        action = teacher_action(task, step_idx)
        prompt = (
            f"Mission brief: {brief}\n"
            f"Constraints: {json.dumps(constraints)}\n"
            f"Telemetry: {json.dumps(public)}\n"
            "Reply with ONE JSON action."
        )
        samples.append({"prompt": prompt, "action": json.dumps(action, separators=(",", ":"))})
        step_resp = checked_json(
            requests.post(f"{OPENENV_BASE_URL}/step", json=action, headers=HEADERS, timeout=30),
            "step",
        )
        step_idx += 1
        if step_resp.get("done"):
            break
        if step_idx > 30:
            break
    score = float(
        checked_json(
            requests.get(f"{OPENENV_BASE_URL}/grader", headers=HEADERS, timeout=30),
            "grader",
        ).get("score", 0.0)
    )
    return samples, score


all_samples, scores = [], []
for rollout_i in range(NUM_ROLLOUTS):
    for task in ("easy", "medium", "hard"):
        samples, score = collect_one_trajectory(task)
        scores.append({"task": task, "score": score})
        if score >= SCORE_KEEP_THRESHOLD:
            all_samples.extend(samples)
        print(
            f"rollout={rollout_i+1}/{NUM_ROLLOUTS} task={task} score={score:.3f} "
            f"kept={len(samples) if score >= SCORE_KEEP_THRESHOLD else 0}"
        )
        time.sleep(0.1)

print("collected", len(all_samples), "training samples")
print("score histogram:", scores[:9], "... total:", len(scores))

rollout=1/6 task=easy score=0.999 kept=7
rollout=1/6 task=medium score=0.666 kept=0
rollout=1/6 task=hard score=0.999 kept=8
rollout=2/6 task=easy score=0.999 kept=7
rollout=2/6 task=medium score=0.666 kept=0
rollout=2/6 task=hard score=0.999 kept=8
rollout=3/6 task=easy score=0.999 kept=7
rollout=3/6 task=medium score=0.666 kept=0
rollout=3/6 task=hard score=0.999 kept=8
rollout=4/6 task=easy score=0.999 kept=7
rollout=4/6 task=medium score=0.666 kept=0
rollout=4/6 task=hard score=0.999 kept=8
rollout=5/6 task=easy score=0.999 kept=7
rollout=5/6 task=medium score=0.666 kept=0
rollout=5/6 task=hard score=0.999 kept=8
rollout=6/6 task=easy score=0.999 kept=7
rollout=6/6 task=medium score=0.666 kept=0
rollout=6/6 task=hard score=0.999 kept=8
collected 90 training samples
score histogram: [{'task': 'easy', 'score': 0.999}, {'task': 'medium', 'score': 0.6663}, {'task': 'hard', 'score': 0.999}, {'task': 'easy', 'score': 0.999}, {'task': 'medium', 'score': 0.6663}, {'task': 'hard', 'score': 

### 1a. Trajectory score distribution (sanity-check plot)

Before we filter by `SCORE_KEEP >= 0.7` and feed the SFT stage, plot the terminal
`/grader` score of every collected trajectory, split by task. This is the first visual
evidence in the run book: *"the teacher actually reaches good scores on easy/hard, and
medium is on the hairy edge of the keep threshold — that's why we filter."*

Saves:

- `trajectory_score_hist.png` — stacked histogram per task with the keep threshold drawn.
- `trajectory_score_summary.json` — the raw numbers behind the bars.

In [ ]:
import json
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Image, display

# Group scores by task so judges can see the teacher's quality per difficulty.
by_task: dict[str, list[float]] = defaultdict(list)
for entry in scores:
    by_task[entry["task"]].append(float(entry["score"]))

hist_path = OUTPUT_DIR / "trajectory_score_hist.png"
summary_path = OUTPUT_DIR / "trajectory_score_summary.json"

fig, ax = plt.subplots(figsize=(7.5, 4.2))
task_order = [t for t in ("easy", "medium", "hard") if t in by_task]
palette = {"easy": "#16a34a", "medium": "#2563eb", "hard": "#dc2626"}
bins = [i / 10 for i in range(11)]  # 0.0 .. 1.0 in 0.1 steps

ax.hist(
    [by_task[t] for t in task_order],
    bins=bins,
    stacked=True,
    label=[t.capitalize() for t in task_order],
    color=[palette.get(t, "#64748b") for t in task_order],
    edgecolor="white",
    linewidth=0.6,
)

threshold = SCORE_KEEP_THRESHOLD
ax.axvline(
    threshold,
    color="#111827",
    linestyle="--",
    linewidth=1.2,
    label=f"keep threshold = {threshold:.2f}",
)
ax.set_xlabel("Terminal grader score (HTTP GET /grader, 0–1)")
ax.set_ylabel("Number of trajectories")
ax.set_title(
    "Teacher trajectory score distribution\n"
    f"{NUM_ROLLOUTS} rollouts × 3 tasks = {len(scores)} episodes; "
    f"kept {sum(1 for s in scores if s['score'] >= threshold)} for SFT"
)
ax.set_xlim(0, 1.0)
ax.grid(axis="y", linestyle=":", alpha=0.4)
ax.legend(loc="upper left", frameon=False, fontsize=9)
fig.tight_layout()
fig.savefig(hist_path, dpi=150)
plt.show()
print("wrote", hist_path)

summary_path.write_text(
    json.dumps(
        {
            "openenv_base_url": OPENENV_BASE_URL,
            "num_rollouts_per_task": NUM_ROLLOUTS,
            "keep_threshold": threshold,
            "scores": scores,
            "by_task_mean": {t: round(sum(v) / len(v), 4) for t, v in by_task.items()},
        },
        indent=2,
    ),
    encoding="utf-8",
)
print("wrote", summary_path)

display(Image(filename=str(hist_path)))

## 2. Build the SFT dataset

Each sample is a chat completion: a single `assistant` turn that emits the JSON action.

In [12]:
from datasets import Dataset
from transformers import AutoTokenizer

if len(all_samples) == 0:
    raise RuntimeError(
        "No samples were collected above SCORE_KEEP_THRESHOLD. "
        "Lower SCORE_KEEP (e.g., 0.3) or debug trajectory generation first."
    )

tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def to_chat(sample):
    messages = [
        {"role": "system", "content": "You are an autonomous logistics Driver. Output ONLY one JSON action."},
        {"role": "user", "content": sample["prompt"]},
        {"role": "assistant", "content": sample["action"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}


random.shuffle(all_samples)
ds = Dataset.from_list(all_samples).map(to_chat, remove_columns=["prompt", "action"])
print("dataset rows:", len(ds))
print("sample text:\n", ds[0]["text"][:600])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/90 [00:00<?, ? examples/s]

dataset rows: 90
sample text:
 <|im_start|>system
You are an autonomous logistics Driver. Output ONLY one JSON action.<|im_end|>
<|im_start|>user
Mission brief: Port labor action has restricted the South hub. You hold two contracts: a VIP block to East and a standard replenishment to South, under a $500 budget. Coordinate both trucks; VIP must not slip while you still honor the secondary drop if funds and time allow.
Constraints: ["Prioritize ORD-HARD-VIP (40 units to East) over the standard order when tradeoffs appear.", "Fulfill both orders when possible without bankrupting the mission (budget must not finish negative).",


## 3. SFT with TRL + LoRA

Tiny LoRA adapter on the small instruction model. ~1–3 minutes on a T4 with the stub data.

In [13]:
import torch
from peft import LoraConfig
from transformers import AutoModelForCausalLM
from trl import SFTConfig, SFTTrainer

device_map = "auto" if torch.cuda.is_available() else None
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

model = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL, torch_dtype=dtype, device_map=device_map, trust_remote_code=True
)

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

kwargs = dict(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="no",
    dataset_text_field="text",
    report_to="none",
)
if torch.cuda.is_available():
    kwargs["bf16"] = True

# TRL version compatibility: some versions expect max_seq_length, some max_length
try:
    sft_config = SFTConfig(max_seq_length=1024, **kwargs)
except TypeError:
    sft_config = SFTConfig(max_length=1024, **kwargs)

# TRL version compatibility: some versions use tokenizer=, others processing_class=
try:
    trainer = SFTTrainer(
        model=model,
        train_dataset=ds,
        args=sft_config,
        peft_config=peft_config,
        processing_class=tokenizer,
    )
except TypeError:
    trainer = SFTTrainer(
        model=model,
        train_dataset=ds,
        args=sft_config,
        peft_config=peft_config,
        tokenizer=tokenizer,
    )

trainer.train()
trainer.save_model(str(OUTPUT_DIR / "driver-lora"))
print("saved adapter to", OUTPUT_DIR / "driver-lora")

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Adding EOS to train dataset:   0%|          | 0/90 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/90 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
5,1.887560
10,1.718556


saved adapter to /content/openenv-driver-sft/driver-lora


## 4. Plots: SFT loss + baseline vs trained grader

Writes into `OUTPUT_DIR` (run **cells 1 → 11 in order**; the loss cell needs `trainer`, then the next cell frees `trainer`/`model` and reloads a fresh base for fair baseline vs LoRA eval). Cells **12 → 15** are the runnable GRPO block and are independent — run them after section 4:

- `sft_loss_curve.png` — training loss vs **optimizer step** (from TRL `log_history`).
- `baseline_vs_trained_grader.png` — **terminal `/grader` score** per task for the **base student** vs **base + LoRA** after SFT (same student rollout harness: one JSON action per step from public telemetry).
- `grader_eval_summary.json` — numeric scores backing the bar chart.
- `grpo_reward_curve.png` — mean GRPO reward vs optimizer step (emitted by section 5 when `GRPO_DRYRUN=0`, the new default).
- `baseline_vs_multi.png` + `rollout_summary.json` — **scripted vs single-agent vs multi-agent** across 3 seeds (emitted by section 6).

Copy these into `docs/plots/` in your repo for the README / writeup.

In [14]:
import matplotlib.pyplot as plt
from IPython.display import Image, display

losses = [(s["step"], s["loss"]) for s in trainer.state.log_history if "loss" in s]
if losses:
    xs, ys = zip(*losses)
    fig, ax = plt.subplots(figsize=(6.5, 3.5))
    ax.plot(xs, ys, marker="o", color="#2563eb")
    ax.set_xlabel("Optimizer step (TRL log_history)")
    ax.set_ylabel("SFT loss (cross-entropy, training batch)")
    ax.set_title(f"Driver SFT loss — {STUDENT_MODEL.split('/')[-1]}")
    ax.grid(axis="y", linestyle=":", alpha=0.45)
    fig.tight_layout()
    plot_path = OUTPUT_DIR / "sft_loss_curve.png"
    fig.savefig(plot_path, dpi=150)
    plt.show()
    print("wrote", plot_path)
    display(Image(filename=str(plot_path)))
else:
    print("no loss logs (dataset too small?)")

wrote /content/openenv-driver-sft/sft_loss_curve.png


In [22]:
import gc
import json

import matplotlib.pyplot as plt
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM

adapter_path = OUTPUT_DIR / "driver-lora"


VALID_ACTION_TYPES = {
    "check_network",
    "load_truck",
    "route_truck",
    "wait",
    "noop",
}


def _coerce_action(obj) -> dict:
    """Normalize a model-generated dict/string into the API's {action_type: ...} schema."""
    if isinstance(obj, str):
        s = obj.strip().lower()
        if s in VALID_ACTION_TYPES:
            return {"action_type": s}
        return {"action_type": "wait", "hours": 1}
    if not isinstance(obj, dict):
        return {"action_type": "wait", "hours": 1}
    d = dict(obj)
    if "action_type" not in d:
        for alt in ("action", "type", "name", "act"):
            if alt in d and isinstance(d[alt], str):
                d["action_type"] = d.pop(alt)
                break
    if "action_type" not in d or d.get("action_type") not in VALID_ACTION_TYPES:
        return {"action_type": "wait", "hours": 1}
    return d


def extract_action_json(text: str) -> dict:
    text = (text or "").strip()
    if not text:
        return {"action_type": "wait", "hours": 1}
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end <= start:
        return _coerce_action(text)
    try:
        parsed = json.loads(text[start : end + 1])
    except json.JSONDecodeError:
        return {"action_type": "wait", "hours": 1}
    return _coerce_action(parsed)


def student_rollout_grader(model_m, task: str, max_steps: int = 32) -> float:
    """One episode: student generates JSON actions from public telemetry until done or cap."""
    reset_resp = checked_json(
        requests.post(
            f"{OPENENV_BASE_URL}/reset", json={"task_level": task}, headers=HEADERS, timeout=30
        ),
        "reset_eval",
    )
    state0 = reset_resp.get("state", {})
    brief = state0.get("mission_brief", "")
    constraints = state0.get("instruction_constraints", [])
    model_m.eval()
    device = next(model_m.parameters()).device

    for _ in range(max_steps):
        public = checked_json(
            requests.get(f"{OPENENV_BASE_URL}/state", headers=HEADERS, timeout=30),
            "state_eval",
        ).get("state", {})
        prompt = (
            f"Mission brief: {brief}\n"
            f"Constraints: {json.dumps(constraints)}\n"
            f"Telemetry: {json.dumps(public)}\n"
            "Reply with ONE JSON action."
        )
        messages = [
            {"role": "system", "content": "You are an autonomous logistics Driver. Output ONLY one JSON action."},
            {"role": "user", "content": prompt},
        ]
        text_in = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text_in, return_tensors="pt", truncation=True, max_length=1024).to(device)
        with torch.no_grad():
            gen = model_m.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
        new_tokens = gen[0, inputs["input_ids"].shape[1] :]
        raw = tokenizer.decode(new_tokens, skip_special_tokens=True)
        action = extract_action_json(raw)
        step_http = requests.post(
            f"{OPENENV_BASE_URL}/step", json=action, headers=HEADERS, timeout=30
        )
        if step_http.status_code != 200:
            # Untrained student often emits actions with bad args (truck_id, route_id, etc.).
            # Treat as a no-op step instead of crashing the whole eval.
            try:
                _ = checked_json(
                    requests.post(
                        f"{OPENENV_BASE_URL}/step",
                        json={"action_type": "wait", "hours": 1},
                        headers=HEADERS,
                        timeout=30,
                    ),
                    "step_eval_fallback",
                )
            except Exception:
                pass
            continue
        step_resp = step_http.json()
        if step_resp.get("done"):
            break

    return float(
        checked_json(requests.get(f"{OPENENV_BASE_URL}/grader", headers=HEADERS, timeout=30), "grader_eval").get(
            "score", 0.0
        )
    )


# Drop training objects so a fresh base copy fits on smaller GPUs.
# Safe-delete: tolerate re-runs / kernel restarts where these names don't exist yet.
for _name in ("trainer", "model"):
    if _name in globals():
        del globals()[_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
_device_map = "auto" if torch.cuda.is_available() else None

base_eval = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL,
    torch_dtype=_dtype,
    device_map=_device_map,
    trust_remote_code=True,
).eval()

tasks_eval = ("easy", "medium", "hard")
print("Evaluating base student (pre-LoRA) …")
baseline_scores = {t: student_rollout_grader(base_eval, t) for t in tasks_eval}
print("baseline grader scores:", baseline_scores)

peft_eval = PeftModel.from_pretrained(base_eval, str(adapter_path)).eval()
print("Evaluating base + LoRA (after SFT) …")
trained_scores = {t: student_rollout_grader(peft_eval, t) for t in tasks_eval}
print("trained grader scores:", trained_scores)

summary_eval = {
    "openenv_base_url": OPENENV_BASE_URL,
    "student_model": STUDENT_MODEL,
    "baseline_grader": baseline_scores,
    "trained_grader": trained_scores,
}
(OUTPUT_DIR / "grader_eval_summary.json").write_text(
    json.dumps(summary_eval, indent=2), encoding="utf-8"
)
print("wrote", OUTPUT_DIR / "grader_eval_summary.json")

labels = [t.capitalize() for t in tasks_eval]
x = list(range(len(labels)))
w = 0.36
fig, ax = plt.subplots(figsize=(7.5, 4.2))
bars_b = ax.bar(
    [i - w / 2 for i in x],
    [baseline_scores[t] for t in tasks_eval],
    width=w,
    label="Base student (pre-LoRA)",
    color="#94a3b8",
)
bars_t = ax.bar(
    [i + w / 2 for i in x],
    [trained_scores[t] for t in tasks_eval],
    width=w,
    label="Base + LoRA (after SFT)",
    color="#16a34a",
)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_xlabel("Task difficulty")
ax.set_ylabel("Terminal grader score (HTTP GET /grader, 0–1 open-interval)")
ax.set_title(
    f"Baseline vs trained — {STUDENT_MODEL.split('/')[-1]}\n"
    "(student generates one JSON action per step; same harness for both bars)"
)
ax.set_ylim(0, 1.05)
ax.legend(loc="upper right", frameon=False)
ax.grid(axis="y", linestyle=":", alpha=0.45)

for bars in (bars_b, bars_t):
    for b in bars:
        h = b.get_height()
        ax.annotate(
            f"{h:.3f}",
            xy=(b.get_x() + b.get_width() / 2, h),
            xytext=(0, 2),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=8,
        )

fig.tight_layout()
_bar_path = OUTPUT_DIR / "baseline_vs_trained_grader.png"
fig.savefig(_bar_path, dpi=150, bbox_inches="tight")
plt.show()
print("wrote", _bar_path)

from IPython.display import Image, display
display(Image(filename=str(_bar_path)))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Evaluating base student (pre-LoRA) …
baseline grader scores: {'easy': 0.001, 'medium': 0.001, 'hard': 0.001}
Evaluating base + LoRA (after SFT) …
trained grader scores: {'easy': 0.001, 'medium': 0.001, 'hard': 0.001}
wrote /content/openenv-driver-sft/grader_eval_summary.json
wrote /content/openenv-driver-sft/baseline_vs_trained_grader.png


## 5. GRPO with `GET /grader` as the reward function

Per the opening-ceremony deck (slides on **Reward & Training Pipeline** and the Sudoku /
Wordle GRPO examples): *"the training loop should connect to your environment, not a
static dataset"*. The cells below do exactly that — they run a short **GRPO** loop on the
SFT-warmed student, where each completion is treated as a candidate JSON action, posted
to the live OpenEnv server, and rewarded with `GET /grader`'s open-interval score (plus
small shaping bonuses for being valid JSON the env accepts).

The default settings are sized to a **T4 end-to-end run** (≈2–4 min):

- `GRPO_DRYRUN=0` (default) → runs `trainer.train()` for `GRPO_STEPS` (default 20) and
  saves `grpo_reward_curve.png` next to the SFT plots. This is the setting judges open.
- `GRPO_DRYRUN=1` → skip weight updates, just verify the reward function returns
  numbers. Useful to confirm wiring without burning GPU hours.

Outputs land in `OUTPUT_DIR`; `grpo_reward_curve.png` joins `sft_loss_curve.png`,
`baseline_vs_trained_grader.png`, and `grader_eval_summary.json` ready to copy into
`docs/plots/`.

In [23]:
import gc
import json
import os
import re
import time
from pathlib import Path

import requests
import torch
from peft import LoraConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

GRPO_TASK = os.environ.get("GRPO_TASK", "easy")
GRPO_STEPS = int(os.environ.get("GRPO_STEPS", "20"))
GRPO_DRYRUN = os.environ.get("GRPO_DRYRUN", "0") == "1"
NUM_GENERATIONS = int(os.environ.get("GRPO_NUM_GENERATIONS", "4"))
MAX_PROMPT_LEN = int(os.environ.get("GRPO_MAX_PROMPT", "512"))
MAX_COMPLETION_LEN = int(os.environ.get("GRPO_MAX_COMPLETION", "96"))

GRPO_OUTPUT_DIR = OUTPUT_DIR / "grpo"
GRPO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def _http_post(path: str, payload: dict, timeout: int = 30) -> dict:
    resp = requests.post(f"{OPENENV_BASE_URL}{path}", json=payload, timeout=timeout)
    resp.raise_for_status()
    return resp.json()


def _http_get(path: str, timeout: int = 30) -> dict:
    resp = requests.get(f"{OPENENV_BASE_URL}{path}", timeout=timeout)
    resp.raise_for_status()
    return resp.json()


def _extract_json(text: str) -> dict | None:
    """Tolerant action-JSON extractor: first balanced {...} block in the completion."""
    if not isinstance(text, str):
        return None
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        return None
    try:
        candidate = json.loads(match.group(0))
        if isinstance(candidate, dict) and "action_type" in candidate:
            return candidate
    except json.JSONDecodeError:
        pass
    return None


def _grader_score(default: float = 0.0) -> float:
    try:
        payload = _http_get("/grader")
    except requests.RequestException:
        return default
    return float(payload.get("score", default))


def _safe_reset(task: str) -> None:
    for attempt in range(3):
        try:
            _http_post("/reset", {"task_level": task})
            return
        except requests.RequestException:
            time.sleep(0.4 * (attempt + 1))


_SHAPING_VALID_JSON = 0.05
_SHAPING_VALID_STEP = 0.05


def grader_reward(prompts, completions, **_kwargs):
    """Reward = GET /grader after a single env step.

    Shaping (small, dense, prevents flat-zero gradients early in training):
      +0.05 if the completion contains a valid action JSON
      +0.05 if /step accepts it (no `error` field)
      + grader.score in (0, 1)
    """
    rewards: list[float] = []
    for completion in completions:
        text = completion if isinstance(completion, str) else getattr(completion, "text", str(completion))
        shaped = 0.0
        action = _extract_json(text)
        if action is None:
            rewards.append(shaped)
            continue
        shaped += _SHAPING_VALID_JSON
        try:
            _safe_reset(GRPO_TASK)
            step_payload = _http_post("/step", action)
            if not step_payload.get("error"):
                shaped += _SHAPING_VALID_STEP
        except requests.RequestException:
            rewards.append(shaped)
            continue
        shaped += _grader_score()
        rewards.append(shaped)
    return rewards


print("Reward function wired:")
print(f"  endpoint     = {OPENENV_BASE_URL}")
print(f"  task         = {GRPO_TASK}")
print(f"  dry-run only = {GRPO_DRYRUN}")
print(f"  steps        = {GRPO_STEPS}")
print(f"  generations  = {NUM_GENERATIONS}")


Reward function wired:
  endpoint     = https://nikitha04-openenv-logistics.hf.space
  task         = easy
  dry-run only = True
  steps        = 20
  generations  = 4


In [24]:
_dry_completions = [
    '{"action_type": "check_network"}',
    '{"action_type": "wait"}',
    '{"action_type": "load_truck", "truck_id": "T101", "warehouse": "North", "amount": 50}',
    "I would call check_network because telemetry is masked.",
]
_dry_rewards = grader_reward(prompts=[""] * len(_dry_completions), completions=_dry_completions)
for completion, reward in zip(_dry_completions, _dry_rewards):
    preview = completion[:60].replace("\n", " ")
    print(f"reward={reward:6.3f}  ←  {preview}")
assert all(isinstance(r, float) for r in _dry_rewards), "reward function must return floats"

if GRPO_DRYRUN:
    print("\nGRPO_DRYRUN=1 → skipping trainer.train(); reward function is wired.")
else:
    print(f"\nGRPO_DRYRUN=0 → launching {GRPO_STEPS} GRPO steps on task={GRPO_TASK} ...")

    # Free any leftover SFT objects so we can host a fresh policy + value model.
    for _name in ("trainer", "model", "base_for_eval", "merged"):
        if _name in globals():
            try:
                del globals()[_name]
            except KeyError:
                pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    grpo_tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL)
    if grpo_tokenizer.pad_token is None:
        grpo_tokenizer.pad_token = grpo_tokenizer.eos_token

    grpo_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    base_policy = AutoModelForCausalLM.from_pretrained(STUDENT_MODEL, torch_dtype=grpo_dtype)
    sft_adapter = OUTPUT_DIR / "sft"
    if sft_adapter.exists():
        policy = PeftModel.from_pretrained(base_policy, str(sft_adapter), is_trainable=True)
    else:
        policy = base_policy

    grpo_lora = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )
    grpo_config = GRPOConfig(
        output_dir=str(GRPO_OUTPUT_DIR),
        learning_rate=5e-6,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        max_steps=GRPO_STEPS,
        logging_steps=1,
        num_generations=NUM_GENERATIONS,
        max_prompt_length=MAX_PROMPT_LEN,
        max_completion_length=MAX_COMPLETION_LEN,
        bf16=torch.cuda.is_available(),
        report_to="none",
    )
    grpo_trainer = GRPOTrainer(
        model=policy,
        args=grpo_config,
        train_dataset=ds,
        processing_class=grpo_tokenizer,
        reward_funcs=[grader_reward],
        peft_config=grpo_lora if not isinstance(policy, PeftModel) else None,
    )
    grpo_trainer.train()
    grpo_trainer.save_model(str(GRPO_OUTPUT_DIR / "adapter"))
    print(f"\nGRPO finished. Adapter saved to {GRPO_OUTPUT_DIR / 'adapter'}")


reward= 0.050  ←  {"action_type": "check_network"}
reward= 0.050  ←  {"action_type": "wait"}
reward= 0.050  ←  {"action_type": "load_truck", "truck_id": "T101", "warehouse
reward= 0.000  ←  I would call check_network because telemetry is masked.

GRPO_DRYRUN=1 → skipping trainer.train(); reward function is wired.


In [25]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Image, display

reward_curve_path = OUTPUT_DIR / "grpo_reward_curve.png"

if GRPO_DRYRUN or "grpo_trainer" not in globals():
    print("Skipping reward-curve plot (GRPO_DRYRUN=1 or trainer was never run).")
else:
    history = grpo_trainer.state.log_history
    rows = [h for h in history if "reward" in h or "rewards/grader_reward" in h]
    steps, rewards = [], []
    for h in rows:
        step = h.get("step")
        reward = h.get("reward", h.get("rewards/grader_reward"))
        if step is None or reward is None:
            continue
        steps.append(int(step))
        rewards.append(float(reward))

    if not steps:
        print("No reward entries logged yet; rerun with more GRPO_STEPS.")
    else:
        fig, ax = plt.subplots(figsize=(7.5, 4.5))
        ax.plot(steps, rewards, marker="o", linewidth=2, color="#1f77b4")
        ax.set_xlabel("GRPO optimizer step")
        ax.set_ylabel("Mean reward (grader.score + shaping, per generation group)")
        ax.set_title(
            f"GRPO reward vs. step — task={GRPO_TASK}, model={STUDENT_MODEL.split('/')[-1]}",
            fontsize=11,
        )
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        fig.savefig(reward_curve_path, dpi=150)
        plt.show()
        print(f"Saved {reward_curve_path}")
        display(Image(filename=str(reward_curve_path)))

        summary_path = OUTPUT_DIR / "grpo_reward_summary.json"
        summary_path.write_text(
            json.dumps(
                {
                    "task": GRPO_TASK,
                    "model": STUDENT_MODEL,
                    "openenv_base_url": OPENENV_BASE_URL,
                    "steps": steps,
                    "rewards": rewards,
                },
                indent=2,
            )
        )
        print(f"Saved {summary_path}")

print(
    "\nNext: copy",
    str(Path("OUTPUT_DIR") / "sft_loss_curve.png") + ",",
    str(Path("OUTPUT_DIR") / "baseline_vs_trained_grader.png") + ",",
    str(Path("OUTPUT_DIR") / "grader_eval_summary.json") + ",",
    "grpo_reward_curve.png, and baseline_vs_multi.png (from section 6) into docs/plots/.",
)


Skipping reward-curve plot (GRPO_DRYRUN=1 or trainer was never run).

Next: copy OUTPUT_DIR/sft_loss_curve.png, OUTPUT_DIR/baseline_vs_trained_grader.png, OUTPUT_DIR/grader_eval_summary.json and (if you ran GRPO) grpo_reward_curve.png into docs/plots/.


## 6. 3-seed scripted vs single-agent vs multi-agent comparison

This is the **headline chart** in the README — the one that shows what the *architecture*
buys you at inference time, independent of training.

For each task difficulty (`easy` / `medium` / `hard`) and 3 random seeds we run three
policies against the same live OpenEnv server and read the terminal `/grader` score:

1. **Scripted optimum** — hand-coded action list from `test_local.py`. No LLM. Ceiling.
2. **Single-agent LLM** — one Driver agent, flat prompt (`inference.run_task`).
3. **Hierarchical multi-agent LLM** — Dispatcher + Driver with a shared message bus and
   a re-plan trigger on adversarial route closures (`inference_multi.run_task`).

Error bars are standard error of the mean across the 3 seeds. The heavy lifting is done
by `training/collect_and_plot.py`; we just clone the repo, set LLM creds, and invoke it
as a module so the output PNG lands in `OUTPUT_DIR` next to the SFT/GRPO plots.

> **Needs `HF_TOKEN` (already set above) for LLM rollouts.** If you want the scripted
> bars only (no API key), pass `--seeds 3` without `--llm`.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

from IPython.display import Image, display

REPO_URL = os.environ.get(
    "OPENENV_REPO_URL",
    "https://huggingface.co/spaces/nikitha04/openenv-logistics",
)
REPO_DIR = Path(os.environ.get("OPENENV_REPO_DIR", "/content/openenv-logistics"))

if not (REPO_DIR / "training" / "collect_and_plot.py").exists():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    print(f"Cloning {REPO_URL} → {REPO_DIR}")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"Repo already present at {REPO_DIR}, skipping clone.")

# `collect_and_plot` imports `inference` / `inference_multi` from the repo root.
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# LLM creds for HF router. HF_TOKEN is already set via notebook login above;
# MODEL_NAME must be one the router actually supports for this account.
os.environ.setdefault("MODEL_NAME", "Qwen/Qwen2.5-7B-Instruct")
os.environ.setdefault("API_BASE_URL", "https://router.huggingface.co/v1")
assert os.environ.get("HF_TOKEN"), (
    "HF_TOKEN must be set for LLM rollouts. Re-run the huggingface_hub.login() cell above."
)

SEEDS = int(os.environ.get("COMPARE_SEEDS", "3"))
USE_LLM = os.environ.get("COMPARE_LLM", "1") == "1"

cmd = [
    sys.executable, "-m", "training.collect_and_plot",
    "--seeds", str(SEEDS),
    "--out", str(OUTPUT_DIR),
]
if USE_LLM:
    cmd.append("--llm")

print(f"\nRunning: {' '.join(cmd)}")
print(f"  cwd           = {REPO_DIR}")
print(f"  OPENENV_BASE  = {os.environ.get('OPENENV_BASE_URL')}")
print(f"  MODEL_NAME    = {os.environ.get('MODEL_NAME')}")
print(f"  seeds         = {SEEDS} (LLM rollouts: {USE_LLM})\n")

result = subprocess.run(cmd, cwd=str(REPO_DIR))
if result.returncode != 0:
    raise RuntimeError(
        f"collect_and_plot exited with code {result.returncode}. "
        "Check LLM creds (HF_TOKEN / MODEL_NAME) or rerun with COMPARE_LLM=0 for the scripted-only bars."
    )

compare_png = OUTPUT_DIR / "baseline_vs_multi.png"
compare_json = OUTPUT_DIR / "rollout_summary.json"
print(f"\nSaved:\n  {compare_png}\n  {compare_json}")

if compare_png.exists():
    display(Image(filename=str(compare_png)))

In [26]:
# Optional helper: download final artifacts to your local machine from Colab.
from pathlib import Path

try:
    from google.colab import files

    for fname in [
        "trajectory_score_hist.png",
        "sft_loss_curve.png",
        "baseline_vs_trained_grader.png",
        "grader_eval_summary.json",
        "grpo_reward_curve.png",
        "grpo_reward_summary.json",
        "baseline_vs_multi.png",
        "rollout_summary.json",
    ]:
        p = OUTPUT_DIR / fname
        if p.exists():
            print("downloading", p)
            files.download(str(p))
        else:
            print("missing", p)
except Exception as exc:
    print("Colab download helper unavailable in this runtime:", exc)
    print("Artifacts are in", OUTPUT_DIR)


downloading /content/openenv-driver-sft/sft_loss_curve.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

downloading /content/openenv-driver-sft/baseline_vs_trained_grader.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

downloading /content/openenv-driver-sft/grader_eval_summary.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

missing /content/openenv-driver-sft/grpo_reward_curve.png
missing /content/openenv-driver-sft/grpo_reward_summary.json
